In [ ]:
# Colab setup: cài dependency nếu runtime chưa có.
import importlib.util, subprocess, sys
from pathlib import Path

missing = [pkg for pkg in ["sentence_transformers", "transformers"] if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "sentence-transformers", "transformers", "accelerate"])

# Mount Google Drive khi chạy trên Colab. Nếu đã mount rồi thì lệnh này chỉ xác nhận lại.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

import json, logging, gc, shutil, zipfile
import numpy as np, torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss, GISTEmbedLoss, MatryoshkaLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator

logging.basicConfig(level=logging.WARNING)

CONTENT_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
DRIVE_DATA = Path("/content/drive/MyDrive/Data")


def unzip_if_needed(zip_path, dest_dir):
    zip_path = Path(zip_path)
    dest_dir = Path(dest_dir)
    target_name = zip_path.stem
    target_dir = dest_dir / target_name
    if target_dir.exists() or not zip_path.exists():
        return
    dest_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest_dir)
    print(f"Unzipped {zip_path} -> {dest_dir}")


# 2 zip bạn đã upload vào MyDrive/Data. Sau khi unzip sẽ nằm ở:
# /content/chunk_outputs_finals và /content/chunk_outputs1_finals
unzip_if_needed(DRIVE_DATA / "chunk_outputs_finals.zip", CONTENT_ROOT)
unzip_if_needed(DRIVE_DATA / "chunk_outputs1_finals.zip", CONTENT_ROOT)


def find_data_dir():
    candidates = [
        CONTENT_ROOT / "training" / "data_preparation_pipeline",
        DRIVE_DATA / "training" / "data_preparation_pipeline",
        DRIVE_DATA,
        Path.cwd() / "training" / "data_preparation_pipeline",
    ]
    for data_dir in candidates:
        if (data_dir / "train_stage1.jsonl").exists() and (data_dir / "train_stage2.jsonl").exists():
            return data_dir
    raise FileNotFoundError(
        "Không tìm thấy train_stage1.jsonl/train_stage2.jsonl. "
        "Hãy upload/copy các file train/test jsonl vào /content/training/data_preparation_pipeline "
        "hoặc /content/drive/MyDrive/Data."
    )


DATA = find_data_dir()
ROOT_DATA = DATA if (DATA / "question.json").exists() else CONTENT_ROOT / "data"
if not (ROOT_DATA / "question.json").exists() and (DRIVE_DATA / "question.json").exists():
    ROOT_DATA = DRIVE_DATA

CHUNK_DIR_INTERNAL = CONTENT_ROOT / "chunk_outputs_finals"
CHUNK_DIR_EXTERNAL = CONTENT_ROOT / "chunk_outputs1_finals"
WORK_DIR = CONTENT_ROOT / "working"
WORK_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    # Colab có mạng nên load trực tiếp từ Hugging Face thay vì folder offline trên Kaggle.
    # BGE-M3 có output 1024d, khớp matryoshka dims [256, 512, 1024].
    "base_model": "BAAI/bge-m3",
    "guide_model": "BAAI/bge-m3",
    "max_seq_len": 512,
    "mrl_dims": [256, 512, 1024],
    "stages": [
        {"name": "stage1", "file": DATA / "train_stage1.jsonl", "epochs": 2, "lr": 2e-6, "batch": 64, "warmup": 5},
        {"name": "stage2", "file": DATA / "train_stage2.jsonl", "epochs": 1, "lr": 5e-7, "batch": 32, "warmup": 5},
    ],
    "eval_file": DATA / "test_dataset.jsonl",
    "use_amp": True,
}

print("Data dir:", DATA)
print("Root data dir:", ROOT_DATA)
print("Internal chunks:", CHUNK_DIR_INTERNAL, "| exists:", CHUNK_DIR_INTERNAL.exists())
print("External chunks:", CHUNK_DIR_EXTERNAL, "| exists:", CHUNK_DIR_EXTERNAL.exists())
print("Work dir:", WORK_DIR)
print("Config OK | torch", torch.__version__, "| CUDA", torch.cuda.is_available())

In [ ]:
def load_examples(path):
    ex = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            q, pos = rec.get("query", "").strip(), rec.get("positive", "").strip()
            if q and pos:
                ex.append(InputExample(texts=[q, pos]))
    return ex


def build_train_evaluator(eval_path):
    queries, corpus, relevant_docs = {}, {}, {}
    with open(eval_path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            idx = rec["index"]
            qid = f"q{idx:04d}"
            queries[qid] = rec["query"]
            relevant_docs[qid] = set()
            for pos in rec.get("positives", []):
                corpus[pos["chunk_id"]] = pos["text"]
                relevant_docs[qid].add(pos["chunk_id"])
            for neg in rec.get("negatives", []):
                if neg.get("type") == "mined" and neg.get("text", "").strip():
                    corpus[f"neg_{idx}_{abs(hash(neg['text'])) % 999999:06d}"] = neg["text"]
    vq = {k: v for k, v in queries.items() if relevant_docs.get(k)}
    vr = {k: v for k, v in relevant_docs.items() if v}
    return InformationRetrievalEvaluator(
        queries=vq,
        corpus=corpus,
        relevant_docs=vr,
        accuracy_at_k=[1, 5, 10],
        mrr_at_k=[10],
        batch_size=128,
        name="train_eval",
        show_progress_bar=False,
    )


guide_model = SentenceTransformer(CFG["guide_model"])
guide_model.max_seq_length = CFG["max_seq_len"]
train_evaluator = build_train_evaluator(CFG["eval_file"])


def train_stage(model, cfg, use_gist, out_dir):
    examples = load_examples(cfg["file"])
    loader = DataLoader(examples, batch_size=cfg["batch"], shuffle=True)
    if use_gist:
        base = GISTEmbedLoss(model=model, guide=guide_model, temperature=0.01)
    else:
        base = MultipleNegativesRankingLoss(model=model, scale=20.0)
    loss = MatryoshkaLoss(model=model, loss=base, matryoshka_dims=CFG["mrl_dims"])
    ckpt = str(Path(out_dir) / "checkpoints" / cfg["name"])
    model.fit(
        train_objectives=[(loader, loss)],
        evaluator=train_evaluator,
        epochs=cfg["epochs"],
        warmup_steps=cfg["warmup"],
        optimizer_params={"lr": cfg["lr"]},
        weight_decay=0.01,
        max_grad_norm=1.0,
        use_amp=CFG["use_amp"] and torch.cuda.is_available(),
        evaluation_steps=len(loader),
        output_path=ckpt,
        save_best_model=True,
        show_progress_bar=True,
    )
    print(f"  {cfg['name']} ({'GIST' if use_gist else 'MNR'}) -> {ckpt}")
    return ckpt


def train_config(out_dir, s1_gist, s2_gist):
    print(f"\n{'=' * 55}\nTRAIN {out_dir} | S1={'GIST' if s1_gist else 'MNR'} S2={'GIST' if s2_gist else 'MNR'}\n{'=' * 55}")
    m = SentenceTransformer(CFG["base_model"])
    m.max_seq_length = CFG["max_seq_len"]
    s1 = train_stage(m, CFG["stages"][0], s1_gist, out_dir)
    del m
    gc.collect()
    torch.cuda.empty_cache()

    m = SentenceTransformer(s1)
    m.max_seq_length = CFG["max_seq_len"]
    s2 = train_stage(m, CFG["stages"][1], s2_gist, out_dir)
    del m
    gc.collect()
    torch.cuda.empty_cache()
    return s2


print("Functions loaded")

In [ ]:
# Mặc định chỉ train cấu hình tốt nhất trước đó: GIST -> MNR.
# Bật RUN_ALL_CONFIGS=True nếu muốn benchmark thêm GIST+GIST và MNR+MNR.
RUN_ALL_CONFIGS = False

CKPT_GG = None
CKPT_MM = None
if RUN_ALL_CONFIGS:
    CKPT_GG = train_config(WORK_DIR / "cfg_gist_gist", s1_gist=True, s2_gist=True)
    CKPT_MM = train_config(WORK_DIR / "cfg_mnr_mnr", s1_gist=False, s2_gist=False)

CKPT_GM = train_config(WORK_DIR / "cfg_gist_mnr", s1_gist=True, s2_gist=False)

print("\n=== Training done ===")
if CKPT_GG:
    print(f"GIST+GIST: {CKPT_GG}")
if CKPT_MM:
    print(f"MNR+MNR:   {CKPT_MM}")
print(f"GIST+MNR:  {CKPT_GM}")

In [ ]:
import json, gc, torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator


def build_corpus(chunk_dir):
    corpus = {}
    for doc_dir in Path(chunk_dir).iterdir():
        if not doc_dir.is_dir():
            continue
        ds = doc_dir.name
        for cf in doc_dir.glob("*.json"):
            recs = json.load(open(cf, encoding="utf-8"))
            if not isinstance(recs, list):
                continue
            for i, rec in enumerate(recs):
                text = str(rec.get("page_content", "")).strip()
                md = rec.get("metadata", {}) or {}
                raw = str(md.get("chunk_id") or f"chunk::{md.get('chunk_index', i)}").strip()
                if not raw or not text:
                    continue
                cid = raw if raw.startswith(f"{ds}::") else f"{ds}::{raw}"
                corpus[cid] = text
    return corpus


def build_internal(p, corpus):
    vq, vr = {}, {}
    for line in open(p, encoding="utf-8"):
        if not line.strip():
            continue
        rec = json.loads(line)
        if rec.get("is_augmented", False):
            continue
        gold = {x["chunk_id"] for x in rec.get("positives", [])} & corpus.keys()
        if not gold:
            continue
        qid = f"q{rec['index']:04d}"
        vq[qid] = rec["query"]
        vr[qid] = gold
    return vq, vr


def build_external(p, corpus):
    vq, vr = {}, {}
    for idx, q in enumerate(json.load(open(p, encoding="utf-8"))):
        gold = set(g for g in q.get("gold_chunk_ids", []) if g in corpus)
        if not gold:
            continue
        qid = f"ext_{q.get('index', idx)}"
        vq[qid] = q["question"]
        vr[qid] = gold
    return vq, vr


def build_btc(p, corpus):
    vq, vr = {}, {}
    for i, line in enumerate(open(p, encoding="utf-8")):
        if not line.strip():
            continue
        rec = json.loads(line)
        gold = set(rec.get("gold_chunk_ids", [])) & corpus.keys()
        if not gold:
            continue
        vq[f"btc_{i:04d}"] = rec["question"]
        vr[f"btc_{i:04d}"] = gold
    return vq, vr


ci = build_corpus(CHUNK_DIR_INTERNAL)
vqi, vri = build_internal(DATA / "test_dataset.jsonl", ci)
ce = build_corpus(CHUNK_DIR_EXTERNAL)
vqe, vre = build_external(ROOT_DATA / "question.json", ce)
vqb, vrb = build_btc(DATA / "question_btc_style.jsonl", ce)
print(f"Internal {len(vqi)}q/{len(ci)} | External {len(vqe)}q/{len(ce)} | BTC {len(vqb)}q/{len(ce)}")

CONFIGS = {"GIST+MNR": CKPT_GM}
if CKPT_GG:
    CONFIGS["GIST+GIST"] = CKPT_GG
if CKPT_MM:
    CONFIGS["MNR+MNR"] = CKPT_MM

TESTS = {
    "INTERNAL": (vqi, vri, ci),
    "EXTERNAL": (vqe, vre, ce),
    "BTC": (vqb, vrb, ce),
}


def ev_dim(path, vq, vr, corpus, name, dim):
    bm = SentenceTransformer(path)
    bm.max_seq_length = 512
    e = InformationRetrievalEvaluator(
        queries=vq,
        corpus=corpus,
        relevant_docs=vr,
        accuracy_at_k=[1, 5, 10],
        mrr_at_k=[10],
        batch_size=128,
        name=name,
        show_progress_bar=False,
        truncate_dim=dim,
    )
    s = e(bm)
    del bm
    gc.collect()
    torch.cuda.empty_cache()
    return (
        s[f"{name}_cosine_accuracy@1"],
        s[f"{name}_cosine_accuracy@5"],
        s[f"{name}_cosine_mrr@10"],
        s[f"{name}_cosine_ndcg@10"],
    )


for test_name, (vq, vr, corpus) in TESTS.items():
    print(f"\n{'=' * 70}\n{test_name} ({len(vq)}q)\n{'=' * 70}")
    print(f"{'Config':<14} {'Dim':>5} {'Acc@1':>7} {'Acc@5':>7} {'MRR@10':>8} {'NDCG@10':>9}")
    print("-" * 56)
    for cfg_name, ckpt in CONFIGS.items():
        for dim in [256, 512, 1024]:
            a1, a5, mrr, ndcg = ev_dim(ckpt, vq, vr, corpus, f"{test_name}_{cfg_name}_{dim}".replace("+", ""), dim)
            print(f"{cfg_name:<14} {dim:>5} {a1 * 100:>6.2f}% {a5 * 100:>6.2f}% {mrr:>8.4f} {ndcg:>9.4f}")

In [ ]:
BASELINES = [
    ("BKAI bi-encoder", "bkai-foundation-models/vietnamese-bi-encoder", 256, None),
    ("BGE-M3 raw", "BAAI/bge-m3", 512, None),
]


def ev_base(path, vq, vr, corpus, name, ms, truncate_dim=None):
    bm = SentenceTransformer(path)
    bm.max_seq_length = ms
    e = InformationRetrievalEvaluator(
        queries=vq,
        corpus=corpus,
        relevant_docs=vr,
        accuracy_at_k=[1, 5, 10],
        mrr_at_k=[10],
        batch_size=128,
        name=name,
        show_progress_bar=False,
        truncate_dim=truncate_dim,
    )
    s = e(bm)
    del bm
    gc.collect()
    torch.cuda.empty_cache()
    return (
        s[f"{name}_cosine_accuracy@1"],
        s[f"{name}_cosine_accuracy@5"],
        s[f"{name}_cosine_mrr@10"],
        s[f"{name}_cosine_ndcg@10"],
    )


for test_name, (vq, vr, corpus) in TESTS.items():
    print(f"\n{'=' * 60}\n{test_name} ({len(vq)}q) - BASELINES\n{'=' * 60}")
    print(f"{'Model':<26} {'Acc@1':>7} {'Acc@5':>7} {'MRR@10':>8} {'NDCG@10':>9}")
    print("-" * 60)
    for label, path, ms, truncate_dim in BASELINES:
        a1, a5, mrr, ndcg = ev_base(path, vq, vr, corpus, f"b_{label[:6]}_{test_name[:2]}", ms, truncate_dim)
        print(f"{label:<26} {a1 * 100:>6.2f}% {a5 * 100:>6.2f}% {mrr:>8.4f} {ndcg:>9.4f}")

In [ ]:
SAVE_DIR = WORK_DIR / "saved_models"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

model_mrl = SentenceTransformer(CKPT_GM)
model_mrl.save(str(SAVE_DIR / "vn_embed_finetuned"))
print(f"Saved: {SAVE_DIR / 'vn_embed_finetuned'}")

zip_base = WORK_DIR / "vn_embed_finetuned"
shutil.make_archive(str(zip_base), "zip", str(SAVE_DIR / "vn_embed_finetuned"))
zip_path = zip_base.with_suffix(".zip")
print(f"Zipped: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1e6:.1f} MB")

# Check mmarco-vi

In [ ]:
import json, numpy as np
from pathlib import Path

MMARCO_PATH = ROOT_DATA / "mmarco_vi_50k.jsonl"

recs = []
for line in open(MMARCO_PATH, encoding="utf-8"):
    if not line.strip():
        continue
    recs.append(json.loads(line))
print(f"Tong mMARCO records: {len(recs)}")
print(f"Fields: {list(recs[0].keys())}")
print(f"Sample: {json.dumps(recs[0], ensure_ascii=False)[:300]}")

In [ ]:
domain_train_path = DATA / "domain_train_final_train.jsonl"

res = []
for line in open(domain_train_path, encoding="utf-8"):
    if not line.strip():
        continue
    res.append(json.loads(line))

print(list(res[0].keys()))

In [ ]:
import json, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "BAAI/bge-reranker-v2-m3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(DEVICE).eval()


@torch.no_grad()
def score_logits(pairs, batch=128):
    out = []
    for i in range(0, len(pairs), batch):
        b = pairs[i:i + batch]
        enc = tok(
            [p[0] for p in b],
            [p[1] for p in b],
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(DEVICE)
        logits = model(**enc).logits.squeeze(-1)
        out.extend(logits.float().cpu().numpy().tolist())
    return np.array(out)


def analyze(path, name):
    recs = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    print(f"\n{name}: {len(recs)} mau (FULL)")
    sp = score_logits([(r["query"], r["positive"]) for r in recs])
    sn = score_logits([(r["query"], r["negative"]) for r in recs])
    m = sp - sn
    print(f"  pos logit: mean {sp.mean():.2f} | range [{sp.min():.1f}, {sp.max():.1f}]")
    print(f"  neg logit: mean {sn.mean():.2f} | range [{sn.min():.1f}, {sn.max():.1f}]")
    print(f"  margin: mean {m.mean():.2f} | std {m.std():.2f}")
    print(f"  % easy (margin>5):  {100 * np.mean(m > 5):.1f}%")
    print(f"  % medium (1-5):     {100 * np.mean((m >= 1) & (m <= 5)):.1f}%")
    print(f"  % hard (0-1):       {100 * np.mean((m > 0) & (m < 1)):.1f}%")
    print(f"  % neg>=pos (m<=0):  {100 * np.mean(m <= 0):.1f}%")
    return m


m_mm = analyze(ROOT_DATA / "mmarco_vi_50k.jsonl", "mMARCO-VI")
m_dm = analyze(DATA / "domain_train_final_train.jsonl", "Domain")
print(f"\n[SLIDE] mMARCO margin {m_mm.mean():.2f} vs Domain {m_dm.mean():.2f}")